# Cross-match Blue Jay galaxies with A3COSMOS

Compare the two catalogues by matching the sources

In [2]:
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy.table import Table, join, hstack
from astropy import units as u
import numpy as np
import os
import pandas as pd

# Load the prior catalogue
prior_path = 'A-COSMOS_prior_v20250312_Gaussian_with_meta_corrected_within_Pbcor_0.1_to_publish.fits'
prior_cat = fits.open(prior_path)[1].data
prior_cat = Table(prior_cat)

# Load the Blue Jay source catalogue
bluejay_path = 'cat_targets.fits'
my_cat = fits.open(bluejay_path)[1].data
my_cat = Table(my_cat)

# Rename columns for cross-matching
my_cat.rename_column('id', 'ID')
my_cat.rename_column('ra', 'RA')
my_cat.rename_column('dec', 'Dec')

# Filter the prior catalogue to ensure detections are reliable and have valid size estimates
neg_flux = prior_cat['Total_flux_pbcor'] < 0
lower = prior_cat['Flag_size_lower_boundary']
upper = prior_cat['Flag_size_upper_boundary']
initial = prior_cat['Flag_size_initial_guess']
zero_flux = prior_cat['Flag_zero_galfit_flux_error']
zero_size = prior_cat['Flag_zero_galfit_size_error']

# Apply quality flagging
alma_cat = prior_cat[
    (neg_flux == False) &
    (lower == False) &
    (upper == False) &
    (initial == False) &
    (zero_flux == False) &
    (zero_size == False)
]

print(f'{len(alma_cat)} out of {len(prior_cat)} sources in the prior catalogue after quality filtering.')

# Keep only the IDs of galaxies that have MIRI photometry
miri_path = '/Users/benjamincollins/University/master/Red_Cardinal/photometry/phot_tables/fits/Phot_Table_MIRI.fits'

miri_table = Table.read(miri_path, format='fits')
galaxy_ids = miri_table['ID']

# Filter out Blue Jay galaxies that are not in the MIRI table
source_cat_miri = my_cat[np.isin(my_cat['ID'], galaxy_ids)]

print(len(source_cat_miri), "galaxies in Blue Jay catalogue after filtering for MIRI photometry.")

# Build coordinates
coords_bluejay = SkyCoord(ra=source_cat_miri['RA']*u.deg, dec=source_cat_miri['Dec']*u.deg)
coords_cosmos = SkyCoord(ra=alma_cat['RA']*u.deg, dec=alma_cat['Dec']*u.deg)

# Match (within 0.3 arcsec, for instance)
idx, d2d, _ = coords_bluejay.match_to_catalog_sky(coords_cosmos)
match_mask = d2d < 0.3 * u.arcsec

print(f"Number of matches with Blue Jay: {np.sum(match_mask)}")

matched_ids = set(source_cat_miri['ID'][match_mask])
print(list(matched_ids))

cosmos_ids = set(alma_cat['ID'][idx[match_mask]])
print(list(cosmos_ids))

2226 out of 341803 sources in the prior catalogue after quality filtering.
111 galaxies in Blue Jay catalogue after filtering for MIRI photometry.
Number of matches with Blue Jay: 4
[np.int64(8280), np.int64(18252), np.int64(21165), np.int64(13103)]
[np.int32(320337), np.int32(765691), np.int32(892054), np.int32(313094)]


Okay now let's try and be reasonable here.

Ideally, we want the data from
- Blue Jay source catalogue (ID, Ra, Dec, z_spec)
- Blue Jay HST/NIRCam photometry (ID, FXXXW_flux, FXXXW_flux_err, Aperture params)
- Blue Jay MIRI photometry (ID, FXXXW_flux, FXXXW_flux_err, Aperture params)
- A3COSMOS ALMA observations (ID_COSMOS, Ra_COSMOS, Dec_COSMOS, Total_flux_pbcor, E_Total_flux_pbcor, E_Total_flux_sim_pbcor and all the rest)

In [11]:
miri_matched = miri_table[match_mask]
miri_matched.rename_column('ID', 'ID_3DHST')
alma_matched = alma_cat[idx[match_mask]]

print(alma_matched['RA'], alma_matched['Dec'])
alma_matched.rename_column('ID', 'ID_COSMOS')







print(alma_matched['Obs_wavelength'])

sky_matched = hstack([alma_matched, miri_matched])

print(sky_matched.columns)


     RA    
-----------
150.0985353
150.0880263
150.0864431
150.1054047    Dec   
---------
2.3653718
2.3951005
2.2643185
2.3128168
Obs_wavelength
--------------
       872.771
       872.804
       1286.74
       872.663
<TableColumns names=('ID_COSMOS','RA','Dec','Total_flux_pbcor','E_Total_flux_pbcor','E_Total_flux_sim_pbcor','Pbcor','Pb_corr_pb_image','Pb_corr_equation','Peak_flux','Image_file','Image_cutout','Obs_wavelength','Primary_beam','RMS_noise','Maj_beam','Min_beam','PA_beam','Galfit_reduced_chi_square','Flag_size_lower_boundary','Flag_size_upper_boundary','Flag_size_initial_guess','Flag_zero_galfit_flux_error','Flag_zero_galfit_size_error','ID_3DHST','MIRI_ap_a','MIRI_ap_b','MIRI_ap_npix','F770W_flux','F770W_flux_err','F770W_abmag','F770W_apflux','F770W_apflux_err','F770W_apflux_errnominal','F770W_apcorr','F770W_bkg','F770W_bkg_err','F770W_qc','F770W_ap_x','F770W_ap_y','F770W_ap_theta','F1000W_flux','F1000W_flux_err','F1000W_abmag','F1000W_apflux','F1000W_apflux_err','F100

In [27]:
from astropy.io import fits
from astroquery.alma import Alma
from astropy.coordinates import SkyCoord
from astropy.table import Table, join, hstack
import astropy.units as u
import numpy as np
from astropy.time import Time

# When A3COSMOS was published
cut_off = Time('2025-03-12')

# Initialise the ALMA service
alma = Alma()

# Load the Blue Jay source catalogue
bluejay_path = 'cat_targets.fits'
my_cat = fits.open(bluejay_path)[1].data
my_cat = Table(my_cat)

# Rename columns for cross-matching
my_cat.rename_column('id', 'ID')
my_cat.rename_column('ra', 'RA')
my_cat.rename_column('dec', 'Dec')

# Keep only the IDs of galaxies that have MIRI photometry
miri_path = '/Users/benjamincollins/University/master/Red_Cardinal/photometry/phot_tables/fits/Phot_Table_MIRI.fits'

miri_table = Table.read(miri_path, format='fits')
galaxy_ids = miri_table['ID']

# Filter out Blue Jay galaxies that are not in the MIRI table
source_cat_miri = my_cat[np.isin(my_cat['ID'], galaxy_ids)]

my_galaxies = source_cat_miri['ID', 'RA', 'Dec']

my_galaxies = Table()
my_galaxies['id'] = my_cat['ID']
my_galaxies['ra'] = my_cat['RA']*u.deg # Explicitly ensure units
my_galaxies['dec'] = my_cat['Dec']*u.deg

# We join 'ivoa.obscore' (the master ALMA table) with your 'TAP_UPLOAD.my_targets'
query = """
SELECT oc.*, mt.id 
FROM ivoa.obscore AS oc 
JOIN TAP_UPLOAD.my_targets AS mt 
ON CONTAINS(POINT('ICRS', oc.s_ra, oc.s_dec), CIRCLE('ICRS', mt.ra, mt.dec, 0.016)) = 1
"""

alma = Alma()
result = alma.query_tap(query, uploads={'my_targets': my_galaxies})

print(result)
x
projects_found = {}

for row in source_cat_miri:
    id = row['ID']
    ra = row['RA']
    dec = row['Dec']

    print("Querying ALMA for galaxy ID:", id, "at RA:", ra, "Dec:", dec)
    galaxy_coord = SkyCoord(ra, dec, unit=(u.deg, u.deg), frame='icrs')

    # Perform a region-based query
    # Searching for any ALMA observations that cover this position (within a 1 arcminute radius)
    result = alma.query_region(galaxy_coord, radius=1*u.arcmin)
    
    time.sleep(1)
    
    # Get valid science data
    science_data = result[result['science_observation'] == True]
    valid_data = science_data[science_data['qa2_passed'] == True]
    
    # Filter useful columns
    audit_table = valid_data['proposal_id', 'obs_release_date', 'obs_id']    
    
    if len(audit_table) == 0:
        print(" ❌ No valid ALMA observations found for this galaxy.")
        continue
    
    # Convert release_date to astropy Time objects for easy comparison
    release_dates = Time(audit_table['obs_release_date'])
    new_data_mask = release_dates > cut_off

    new_proposals = audit_table[new_data_mask]
    
    unique_ids = np.unique(new_proposals['proposal_id'])
    
    projects_found[id] = unique_ids
    
    print(f" ✅ Found {len(unique_ids)} new ALMA project(s) for galaxy ID {id}: {unique_ids}")


DALServiceError: 502 Server Error: Proxy Error for url: https://almascience.nrao.edu/tap/sync/yonysbunbbtoz4gd/run

In [15]:
print(result.columns)
print(result['member_ous_uid'], result['obs_title'], result['publication_year'])


obs_id = result[result['publication_year'] > 2024]
obs_id = obs_id['obs_release_date']
print(obs_id)


<TableColumns names=('pol_states','obs_publisher_did','obs_collection','facility_name','instrument_name','obs_id','dataproduct_type','calib_level','collections','target_name','s_ra','s_dec','s_fov','s_region','s_xel1','s_xel2','em_xel','t_xel','pol_xel','s_resolution','t_min','t_max','t_exptime','t_resolution','em_min','em_max','em_res_power','sensitivity_10kms','cont_sensitivity_bandwidth','pwv','group_ous_uid','member_ous_uid','asdm_uid','obs_title','type','scan_intent','science_observation','spatial_scale_max','qa2_passed','bib_reference','science_keyword','scientific_category','pi_userid','pi_name','spectral_resolution','lastModified','o_ucd','access_url','access_format','access_estsize','proposal_id','data_rights','gal_longitude','gal_latitude','band_list','em_resolution','bandwidth','antenna_arrays','is_mosaic','obs_release_date','spatial_resolution','frequency_support','frequency','velocity_resolution','obs_creator_name','pub_title','first_author','authors','pub_abstract','publi